# <font color='blue'>Pipelines de CI/CD Para Operações de Machine Learning e IA</font>
## <font color='blue'>Projeto 1</font>
### <font color='blue'>MLOps Nível 0 - Processo Manual de Machine Learning Para Prever o Comportamento do Consumidor</font>

## Instalação, Carregamento e Gerenciamento de Pacotes

In [ ]:
# Para atualizar um pacote, execute o comando abaixo no terminal ou prompt de comando:
# pip install -U nome_pacote

# Para instalar a versão exata de um pacote, execute o comando abaixo no terminal ou prompt de comando:
# pip install nome_pacote==versão_desejada

# Depois de instalar ou atualizar o pacote, reinicie o jupyter notebook.

# Instala o pacote watermark. 
# Esse pacote é usado para gravar as versões de outros pacotes usados neste jupyter notebook.
!pip install -q -U watermark

In [ ]:
%env TF_CPP_MIN_LOG_LEVEL=3

In [ ]:
!pip install -q tensorflow==2.15

In [ ]:
# Imports

# Imports para manipulação e visualização de dados
import numpy as np
import pandas as pd
import matplotlib as m
import matplotlib.pyplot as plt
import seaborn as sns

# Imports para pré-processamento e avaliação
import sklearn
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import MinMaxScaler
from sklearn.metrics import accuracy_score

# Imports para Deep Learning
import tensorflow as tf
import keras
from keras.models import Sequential
from keras.layers import Dense, Input, Dropout
from keras.losses import categorical_crossentropy
from keras.utils import to_categorical
from keras.callbacks import EarlyStopping, ReduceLROnPlateau
from keras.optimizers import Nadam

In [ ]:
# Versões dos pacotes usados neste jupyter notebook
%reload_ext watermark
%watermark 

## Dados Para Treinamento e Avaliação do Modelo

In [ ]:
# Carrega o dataset
dataset = pd.read_csv("dataset.csv")

In [ ]:
# Shape
dataset.shape

In [ ]:
# Visualiza os dados
dataset.head(5)

**Vamos usar Inteligência Artificial, especificamente Deep Learning, para prever o cancelamento de reservas em hotéis.**

## Quando é Necessário Fazer Análise Exploratória?

In [ ]:
# Número de colunas
print(len(dataset.columns))

In [ ]:
# Tipos de dados
dataset.dtypes

In [ ]:
# Vamos checar quais colunas são do tipo string
for column, series in dataset.items():
    if str(type(series[0])) == "<class 'str'>":
        print(column)

Não podemos levar valores no formato de texto para o treinamento do modelo. Vamos tratar isso daqui a pouco. Primeiro vamos checar se temos valores ausentes.

In [ ]:
# Total de valores ausentes por coluna
dataset.isnull().sum()

Sim, temos em algumas colunas. Então vamos ao processo de limpeza e transformação dos dados.

## Pipeline de Limpeza e Transformação

A coluna company tem quase todos os registros nulos e não faz sentido manter essa coluna. Vamos removê-la.

In [ ]:
# Remove a coluna "company"
dataset = dataset.drop(['company'], axis = 1)

Temos ainda duas colunas com valores nulos e diversas estratégias poderiam ser adotadas. Valores ausentes não podem ser deixados no dataset.

Para os valores ausentes na coluna "country", vamos apenas remover a coluna. Para a coluna "agent", removeremos todas as linhas onde esta coluna for valor ausente.

Você pode testar outras abordagens se desejar.

In [ ]:
# Remove a coluna "country"
dataset = dataset.drop(['country'], axis = 1)

In [ ]:
# Removemos valores ausentes que ainda estverem no dataset
dataset = dataset.dropna()

In [ ]:
# Shape
dataset.shape

In [ ]:
# Total de valores ausentes por coluna
dataset.isnull().sum()

Agora vamos trabalhar em outras variáveis. Temos muitas variáveis categóricas nesse dataset e teremos que tomar algumas ações. 

Uma alternativa simples é aplicar o Label Encoding, ou seja, convertemos o texto em cada coluna para um valor numérico correspondente. Vamos ao trabalho.

**Encoding da variável com o mês de chegada no hotel.**

In [ ]:
# Valores únicos para a coluna de mês de chegada
dataset.arrival_date_month.unique()

In [ ]:
# Dicionário de meses
mes_dict = {"January": 1,
            "February": 2,
            "March": 3,
            "April": 4,
            "May": 5, 
            "June": 6,
            "July": 7,
            "August": 8,
            "September": 9,
            "October": 10,
            "November": 11,
            "December": 12}

In [ ]:
# Vamos converter os meses para o valor numérico correspondente, conforme colocamos no dicionário
dataset["arrival_date_month_numerical"] = dataset.arrival_date_month.apply(lambda x: mes_dict[x])

In [ ]:
# Total de registros para cada mês
dataset.arrival_date_month_numerical.value_counts()

Mês de Agosto teve o maior número de reservas e mês de Janeiro o menor. Isso se explica porque no hemisfério Norte o auge do Verão e as férias são em Agosto.

In [ ]:
# Não precisamos mais da coluna "arrival_date_month"
dataset = dataset.drop("arrival_date_month", axis = 1)

**Encoding das variáveis com tipo de quarto reservado e tipo de quarto ocupado.**

In [ ]:
# Valores únicos de quartos reservados
dataset.reserved_room_type.unique()

In [ ]:
# Valores únicos de quartos já ocupados
dataset.assigned_room_type.unique()

Vamos adotar a mesma estratégia usada com os meses.

In [ ]:
# Dicionário para o encoding do tipo de quarto
tipo_quarto_dict = {"A": 1, "B": 2, "C": 3, "D": 4, "E": 5, "F": 6, "G": 7, "H": 8, "I": 9, "K": 11, "L": 12, "P": 15}

In [ ]:
# Atribuímos o dicionário para fazer o encoding nas colunas com texto e criar as colunas numéricas
dataset["reserved_room_type_numerical"] = dataset.reserved_room_type.apply(lambda x: tipo_quarto_dict[x])
dataset["assigned_room_type_numerical"] = dataset.assigned_room_type.apply(lambda x: tipo_quarto_dict[x])

In [ ]:
# Visualiza os dados
dataset.head()

Removemos as duas colunas com valores em texto.

In [ ]:
# Não precisamos mais das colunas "reserved_room_type" e "assigned_room_type"
dataset = dataset.drop(["reserved_room_type", "assigned_room_type"], axis = 1)

**Encoding da variável com o tipo de depósito.**

In [ ]:
# Valores únicos da variável "deposit_type"
dataset.deposit_type.unique()

In [ ]:
# Dicionário
tipo_deposito_dict = {"No Deposit": 0, "Refundable": 1, "Non Refund": -1}

In [ ]:
# Aplicamos o encoding
dataset["deposit_type_numerical"] = dataset.deposit_type.apply(lambda x: tipo_deposito_dict[x])

In [ ]:
# Drop da coluna "deposit_type"
dataset = dataset.drop("deposit_type", axis = 1)

**Encoding da variável com o tipo de cliente.**

In [ ]:
# Valores únicos
dataset.customer_type.unique()

In [ ]:
# Dicionário
tipo_cliente_dict = {"Transient": 0, "Contract": 1, "Transient-Party": 2, "Group":3}

In [ ]:
# Aplicamos o encoding
dataset["customer_type_numerical"] = dataset.customer_type.apply(lambda x: tipo_cliente_dict[x])

In [ ]:
# Drop da coluna "customer_type"
dataset = dataset.drop("customer_type", axis = 1)

**Encoding da variável com o tipo de hotel.**

In [ ]:
# Valores únicos
dataset.hotel.unique()

In [ ]:
# Dicionário com o tipo de hotel
tipo_hotel_dict = {"Resort Hotel": 0, "City Hotel": 1}

In [ ]:
# Aplica o encoding e cria nova coluna
dataset["hotel_numerical"] = dataset.hotel.apply(lambda x: tipo_hotel_dict[x])

In [ ]:
# Drop da coluna "hotel"
dataset = dataset.drop("hotel", axis = 1)

**Encoding da variável com o tipo de refeição.**

In [ ]:
# Valores únicos
dataset.meal.unique()

In [ ]:
# Dicionário 
tipo_refeicao_dict = {"BB": 0, "FB": 1, "HB": 2, "SC": 3, "Undefined": -1}

In [ ]:
# Aplica o encoding e cria nova coluna
dataset["meal_numerical"] = dataset.meal.apply(lambda x: tipo_refeicao_dict[x])

In [ ]:
# Drop da coluna "meal"
dataset = dataset.drop("meal", axis = 1)

**Encoding da variável com o tipo de segmento de mercado.**

In [ ]:
# Valores únicos
dataset.market_segment.unique()

In [ ]:
# Valores únicos
dataset.distribution_channel.unique()

In [ ]:
# Dicionário
segmento_mercado_dict = {"Direct": 0,
                         "Corporate": 1,
                         "Online TA": 2,
                         "Offline TA/TO": 3,
                         "Complementary": 4,
                         "Groups": 5,
                         "Aviation": 6,
                         "GDS": 7,
                         "TA/TO": 8,
                         "Undefined": -1}

In [ ]:
# Aplica o encoding
dataset["market_segment_numerical"] = dataset.market_segment.apply(lambda x: segmento_mercado_dict[x])
dataset["distribution_channel_numerical"] = dataset.distribution_channel.apply(lambda x: segmento_mercado_dict[x])

In [ ]:
# Não precisamos mais das colunas "market_segment" e "distribution_channel"
dataset = dataset.drop(["market_segment", "distribution_channel"], axis = 1)

In [ ]:
# Visualiza os dados
dataset.head(5)

In [ ]:
# Tipos de dados
dataset.dtypes

Temos ainda 2 variáveis do tipo object, ou seja, string. Como as duas variáveis contém status de reserva e data, vamos apenas removê-las.

In [ ]:
# Valores únicos (esta será nossa variável alvo, o que desejamos prever)
dataset.is_canceled.unique()

In [ ]:
# Valores únicos
dataset.reservation_status.unique()

In [ ]:
# Drop
dataset = dataset.drop("reservation_status", axis = 1)

In [ ]:
# Valores únicos
dataset.reservation_status_date.unique()

In [ ]:
# Drop
dataset = dataset.drop("reservation_status_date", axis = 1)

In [ ]:
# Shape
dataset.shape

In [ ]:
# Visualiza os dados
dataset.head(5)

In [ ]:
# Tipos de dados
dataset.dtypes

Limpeza concluída, podemos preparar os dados para treinamento. Vamos seguir para o pré-processamento dos dados.

## Pipeline de Pré-Processamento dos Dados

Primeiro separamos o dataset em X e y.

In [ ]:
# Preparamos os dados de entrada em x
X = dataset.drop(["is_canceled"], axis = 1)

In [ ]:
# Preparamos a variável de saída y
y = dataset["is_canceled"]

In [ ]:
# Convertemos para o tipo categórico
y = to_categorical(y, num_classes = None)

Dividimos os dados em treino e teste, com proporção 80/20.

In [ ]:
# Divisão em treino e teste
X_treino, X_teste, y_treino, y_teste = train_test_split(X, y, shuffle = True, test_size = 0.2, random_state = 420)

In [ ]:
# Shape
print("Shape de X_treino =", X_treino.shape)
print("Shape de y_treino =", y_treino.shape)
print("Shape de X_teste =", X_teste.shape)
print("Shape de y_teste =", y_teste.shape)

In [ ]:
X_treino.info()

Como as variáveis estão em escala diferente, vamos padronizar e deixar tudo na mesma escala. Fazemos isso somente com X.

In [ ]:
# Criamos o objeto scaler
scaler = MinMaxScaler()

In [ ]:
# Treinamos e Aplicamos o scaler aos dados de entrada de treino
X_treino = scaler.fit_transform(X_treino)

In [ ]:
# Aplicamos o scaler aos dados de entrada de teste
X_teste = scaler.transform(X_teste)

In [ ]:
# Resumo
print(f'Matriz da Variável de Entrada:\t{X.shape}\nMatriz da Variável de Saída:\t{y.shape}')

In [ ]:
X_treino

In [ ]:
X_teste

## Pipeline de Construção do Modelo

Vamos criar o modelo de Deep Learning com 5 camadas densas, camadas de dropout para evitar o overfitting e ativação softmax para a classificação.

In [ ]:
X.shape

In [ ]:
# Criamos o modelo
modelo = Sequential()
modelo.add(Dense(200, input_dim = X.shape[1], activation = 'relu'))
modelo.add(Dropout(0.1))
modelo.add(Dense(200, activation = 'relu'))
modelo.add(Dropout(0.2))    
modelo.add(Dense(200, activation = 'relu'))
modelo.add(Dropout(0.2))
modelo.add(Dense(100, activation = 'relu'))
modelo.add(Dropout(0.1))
modelo.add(Dense(2, activation = 'softmax'))

## Pipeline de Otimização e Compilação do Modelo

In [ ]:
# Usaremos como otimizador o algoritmo Nadam pois o conjunto de dados é complexo
# https://keras.io/api/optimizers/Nadam/
otimizador = tf.keras.optimizers.legacy.Nadam(learning_rate = 0.0001, 
                                              beta_1 = 0.9, 
                                              beta_2 = 0.999, 
                                              epsilon = 1e-07)

In [ ]:
# Compilamos o modelo
modelo.compile(optimizer = otimizador, loss = 'categorical_crossentropy', metrics = ['accuracy'])

In [ ]:
# Vamos criar 2 callbacks
# Um para finalizar o treinamento se depois de 20 passadas o erro não mudar
# Outro para reduzir a taxa de aprendizagem se o erro não mudar após 5 passadas
callbacks = [EarlyStopping(monitor = "loss", patience = 20), 
             ReduceLROnPlateau(monitor = "loss", patience = 5)]

## Pipeline de Treino e Avaliação do Modelo

In [ ]:
# Hiperparâmetros
num_epochs = 50
batch_size = 32

> Treinamento do Modelo.

In [ ]:
%%time

print("\nTreinamento Iniciado.\n")

history = modelo.fit(X_treino, 
                         y_treino, 
                         epochs = num_epochs, 
                         batch_size = batch_size, 
                         callbacks = callbacks)

print("\nTreinamento Concluído.\n")

> Avaliação do Modelo nos Dados de Treino.

O treinamento ocorreu com sucesso. Vamos agora avaliar o modelo.

In [ ]:
# Plot da Acurácia em Treino
plt.figure(figsize = [10,8])
plt.title("Curva de Aprendizado do Modelo - Acurácia")
plt.plot(history.history['accuracy'], label = 'Acurácia em Treino')
plt.xlabel("Epochs")
plt.legend()
plt.grid()

In [ ]:
# Plot do Erro em Treino
plt.figure(figsize = [10,8])
plt.title("Curva de Aprendizado do Modelo - Erro")
plt.plot(history.history['loss'], label = 'Erro em Treino')
plt.xlabel("Epochs")
plt.legend()
plt.grid()

> Avaliação do Modelo nos Dados de Teste.

In [ ]:
# Fazemos as previsões com os dados de teste
previsoes = modelo.predict(X_teste)

In [ ]:
# Convertemos as previsões em previsões de classe
y_pred = (previsoes > 0.5) 

In [ ]:
# Calculamos a acurácia comparando valor real com valor previsto
accuracy_score(y_teste, y_pred)

Nosso modelo tem uma performance de 84%. Um bom trabalho. Mas talvez alguns ajustes possam ser feitos no modelo para melhorar essa performance. Se estiver satisfeito com a performance do modelo então você pode salvar o padronizador e o modelo:

In [ ]:
# Salva o scaler
import joblib
joblib.dump(scaler, 'scaler.pkl')

In [ ]:
# Salva o modelo
modelo.save("modelo.keras")

## Deploy e Pipeline de Inferência com o Modelo Treinado

Vamos considerar que os dados já atendem aos requisitos do pipeline de limpeza e transformação!

In [ ]:
# Novos dados de um novo cliente
# Não temos a variável alvo. Isso é o que o modelo irá prever
novos_dados = {
    'lead_time': [15.00],
    'arrival_date_year': [2016.00],
    'arrival_date_week_number': [23.00],
    'arrival_date_day_of_month': [30.00],
    'stays_in_weekend_nights': [2.00],
    'stays_in_week_nights': [5.00],
    'adults': [2.00],
    'children': [0.00],
    'babies': [0.00],
    'is_repeated_guest': [0.00],
    'previous_cancellations': [0.00],
    'previous_bookings_not_canceled': [0.00],
    'booking_changes': [0.00],
    'agent': [14.00],
    'days_in_waiting_list': [0.00],
    'adr': [115.84],
    'required_car_parking_spaces': [0.00],
    'total_of_special_requests': [0.00],
    'arrival_date_month_numerical': [5.00],
    'reserved_room_type_numerical': [4.00],
    'assigned_room_type_numerical': [4.00],
    'deposit_type_numerical': [0.00],
    'customer_type_numerical': [0.00],
    'hotel_numerical': [1.00],
    'meal_numerical': [0.00],
    'market_segment_numerical': [0.00],
    'distribution_channel_numerical': [0.00]
}

In [ ]:
# Criando o DataFrame
df_novo_cliente = pd.DataFrame(novos_dados)

In [ ]:
df_novo_cliente.shape

In [ ]:
df_novo_cliente

In [ ]:
# Carregando o scaler do disco
scaler_final = joblib.load('scaler.pkl')

In [ ]:
# Aplicamos o scaler aos novos_dados
df_novo_cliente_array = scaler_final.transform(df_novo_cliente)

In [ ]:
df_novo_cliente_array

In [ ]:
# Carrega o modelo do disco
from tensorflow.keras.models import load_model
modelo_final = load_model("modelo.keras")

In [ ]:
# Fazemos a previsão
previsao = modelo_final.predict(df_novo_cliente_array)

In [ ]:
previsao

> Podemos entregar a previsão no formato de probabilidade (quando o modelo permitir):

In [ ]:
print("Esta é a probabilidade do cliente não cancelar a reserva (classe 0):", previsao[0, 0] * 100)

In [ ]:
print("Esta é a probabilidade do cliente cancelar a reserva (classe 1):", previsao[0, 1] * 100)

> Ou podemos entregar o modelo no formato de previsão de classe (para modelos de classificação):

In [ ]:
# Verificando o primeiro valor do array
if previsao[0, 0] > 0.50:
    print("Esse cliente provavelmente não vai cancelar a reserva!")
else:
    print("Esse cliente provavelmente vai cancelar a reserva!")

In [ ]:
%reload_ext watermark
%watermark

In [ ]:
#%watermark -v -m

In [ ]:
#%watermark --iversions

# Fim